# 5. Local development with `DefaultAzureCredential`

The question every Azure-adjacent developer asks on day one: **"How do I run this thing on my laptop if the real auth only works in Azure?"**

Answer: `DefaultAzureCredential` from `azure-identity`. It's a **credential chain** that walks through multiple credential sources and uses the first one that works. That's why the exact same code:

```python
from azure.identity import DefaultAzureCredential
cred = DefaultAzureCredential()
token = cred.get_token('https://storage.azure.com/.default').token
```

… works on your laptop *and* in Container Apps *and* in GitHub Actions *and* in Azure Functions. No `if` branches. No `#ifdef LOCAL`.

## The credential chain (in order)

1. **EnvironmentCredential** — looks for `AZURE_CLIENT_ID` + (`AZURE_CLIENT_SECRET` or cert) + `AZURE_TENANT_ID`.
2. **WorkloadIdentityCredential** — federated tokens (AKS workload identity, GitHub Actions OIDC).
3. **ManagedIdentityCredential** — talks to IMDS — only works inside Azure compute.
4. **SharedTokenCacheCredential** — *legacy and deprecated*; cached tokens from older MS tooling. Exclude it.
5. **VisualStudioCodeCredential** — *deprecated*; depends on an "Azure Account" extension that no longer ships a usable cache. Exclude it and use `az login`.
6. **AzureCliCredential** — whatever `az login` gave you.
7. **AzurePowerShellCredential** — same for `Connect-AzAccount`.
8. **AzureDeveloperCliCredential** — the newer `azd` CLI.

Each one is tried in order. First success wins.

**Do not ship `DefaultAzureCredential()` bare to production.** In a deployed service you know
exactly which credential should be used, and walking the chain has two real costs: every
miss adds latency and log noise (`ManagedIdentityCredential` in particular can wait on an
IMDS timeout), and a chain that silently falls through to a *developer's* cached CLI login is
how a personal identity ends up running production workloads. Either pin it —

```python
from azure.identity import ManagedIdentityCredential
cred = ManagedIdentityCredential(client_id=os.environ['AZURE_CLIENT_ID'])  # user-assigned MI
```

— or narrow the chain with the `exclude_*` flags:

```python
DefaultAzureCredential(
    exclude_shared_token_cache_credential=True,
    exclude_visual_studio_code_credential=True,
    exclude_interactive_browser_credential=True,   # already the default; be explicit
)
```

Use the full chain for local dev convenience; pin the credential in the deployment.

## The three local-dev patterns (pick one)

### Pattern A — Developer identity (recommended for humans)

```bash
az login
# then run your app
```

`DefaultAzureCredential` skips to `AzureCliCredential`, which exchanges your logged-in principal for a token. **You** call the API as **you**. Grant your user account permissions in dev.

### Pattern B — Local service principal (for reproducing prod app-only flows)

```bash
export AZURE_CLIENT_ID=<app-registration-id>
export AZURE_CLIENT_SECRET=<secret>
export AZURE_TENANT_ID=<tenant-id>
```

`EnvironmentCredential` picks these up first. Secret goes in `.env` (git-ignored) or 1Password / macOS Keychain — *not* in source.

### Pattern C — Emulator / mock (this lab)

When the real Azure dependency isn't available (or you want offline/CI speed), point the SDK at a mock that speaks the same protocols. We ship one in `fake-entra/`.

Most teams mix A + B: humans use `az login`, CI pipelines use service principals with OIDC federation (no stored secret).

In [ ]:
# Walk through the chain conceptually. We do NOT instantiate DefaultAzureCredential here:
# azure-identity refuses non-HTTPS authorities and our mock is plain HTTP, so any call
# would fail for a reason that has nothing to teach. In real Azure the two lines below
# are all you'd need:
#
#     from azure.identity import DefaultAzureCredential
#     token = DefaultAzureCredential().get_token('api://api-b/.default').token
#
# Below we just *set* the env vars EnvironmentCredential would pick up and confirm
# which source would win. Nothing here contacts Entra, real or mock.
import logging
import os
import sys

os.environ['AZURE_CLIENT_ID']     = 'daemon-client-id'
os.environ['AZURE_CLIENT_SECRET'] = 'daemon-secret-value'
os.environ['AZURE_TENANT_ID']     = 'contoso'

_ENV_VARS = ['AZURE_CLIENT_ID', 'AZURE_CLIENT_SECRET', 'AZURE_TENANT_ID']
present = {k: k in os.environ for k in _ENV_VARS}
print('EnvironmentCredential inputs present:', present)
assert all(present.values()), f'EnvironmentCredential needs all of {_ENV_VARS}'
print('-> DefaultAzureCredential would use EnvironmentCredential (first match).')

# --- Turning on the logging you will need the first time this misbehaves --------------
# DefaultAzureCredential reports only the LAST failure in the chain. These two lines make
# every attempt visible, which is the difference between a five-minute fix and an afternoon.
logging.basicConfig(stream=sys.stdout, level=logging.WARNING)
logging.getLogger('azure.identity').setLevel(logging.DEBUG)
print('azure.identity log level ->',
      logging.getLevelName(logging.getLogger('azure.identity').level))
assert logging.getLogger('azure.identity').level == logging.DEBUG

# NOTE: pass logging_enable=True to the credential as well if you also want the raw HTTP
# traffic -- DefaultAzureCredential(logging_enable=True). That logs headers, so never
# leave it on in a deployed service: it will write bearer tokens into your log store.

> We don't actually `get_token()` against real Entra here because that would require an Azure tenant. The point is: **same code**. In real Azure you'd call `cred.get_token('api://api-b/.default')` and get back a real token from Entra; the FastAPI `Authorization: Bearer` call that follows is identical.

## Wiring `DefaultAzureCredential` into a FastAPI client

Typical snippet you'd drop into `api-a` (or any caller):

```python
from azure.identity import DefaultAzureCredential
import httpx

_cred = DefaultAzureCredential()
_scope = 'api://api-b/.default'

def call_api_b(path: str) -> dict:
    token = _cred.get_token(_scope).token
    r = httpx.get(f'{API_B_URL}{path}', headers={'Authorization': f'Bearer {token}'}, timeout=5)
    r.raise_for_status()
    return r.json()
```

`cred.get_token()` is cached internally — it only hits Entra when the cached token is near expiry. Safe to call on every request.

## Things that bite people

- **Multi-tenant local dev**: `az login` gives you a token for your *default* tenant. Use `az login --tenant <id>` or `AZURE_TENANT_ID` env var.
- **Stale token cache**: if you rotate a secret and DefaultAzureCredential keeps using the old one, kill any local `msal` / `azure-cli` token caches in `~/.azure/` and `~/.IdentityService/`.
- **VS Code → az CLI mismatch**: disable `VisualStudioCodeCredential` explicitly (it is deprecated anyway) so you know the token came from `az login`.
- **`get_token` is cached, but the cache is per-credential-object**: build the credential once at module scope. A `DefaultAzureCredential()` per request means a full chain walk per request.
- **Logs hide the real error**: DefaultAzureCredential *only* reports the last failure in the chain. Turn on `azure.identity` DEBUG logging (the cell above does exactly that) to see every attempt.
- **Scopes vs resources**: v1 endpoints use `resource=<uri>`, v2 uses `scope=<uri>/.default`. Mixing them gives cryptic 400s.

## CI/CD without secrets — workload identity federation

Instead of storing `AZURE_CLIENT_SECRET` in GitHub Actions, you can configure federated credentials: GitHub mints a short-lived OIDC JWT, Entra trusts it, you get a token. Secret-less CI.

```yaml
- uses: azure/login@v2
  with:
    client-id: ${{ secrets.AZURE_CLIENT_ID }}
    tenant-id: ${{ secrets.AZURE_TENANT_ID }}
    subscription-id: ${{ secrets.AZURE_SUBSCRIPTION_ID }}
    # no client-secret!
```

## Summary

- `DefaultAzureCredential` = one credential for all environments — great for local dev,
  but **pin the credential (or narrow the chain) in deployed services**.
- Local: `az login` (Pattern A) or env vars (Pattern B).
- CI: federated identity, no secrets in vaults.
- Azure: managed identity, no secrets anywhere.
- Build the credential once; `get_token()` caches on the object.
- Turn on `azure.identity` DEBUG logging first when a credential misbehaves — but never
  `logging_enable=True` in production, it logs bearer tokens.